# E2 · YOLOv8n + CLIP: recortar el animal antes de embeber

**Pregunta.** Si se elimina el fondo recortando al animal, ¿el embedding mide mejor la identidad que con la imagen completa (E1)?

| | |
|---|---|
| Pipeline | YOLOv8n (COCO, clases gato/perro) detecta al animal → recorte de la mejor caja con 10 % de margen (sin detección: imagen completa) → el mismo CLIP ViT-B/32 de E1 |
| Entrenamiento | ninguno (*zero-shot*); solo cambia el preprocesamiento respecto de E1 |
| Modelo | `pettrace-embedder` versión 2 (`clip-vit-b32-crop@v1`, alias `@candidate`); mismo encoder que E1 + `detector.onnx` |
| Dataset | `pettrace-reid@v3`, mismo protocolo que E1 |

Requiere haber ejecutado E1 (la comparación lee `reports/e1_v3`). Reproducir: `make notebook NB=notebooks/e2_yolo_crop_zero_shot.ipynb`.

In [ ]:
import json, os

# Paths are relative to the repository root (nbconvert starts the kernel in notebooks/).
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR", "false")
os.environ.setdefault("TQDM_DISABLE", "1")
import warnings
warnings.filterwarnings("ignore", message="IProgress not found")
from collections import Counter

import mlflow
import numpy as np
import onnxruntime as ort
import yaml
from IPython.display import Image, Markdown, display
from PIL import Image as PILImage, ImageOps

from reid import data, metrics
from reid.evaluation import evaluate, load_model, log_run
from reid.figures import explain, metrics_table, save

print("código  :", os.environ.get("GIT_SHA", "?")[:7], "(sin cambios locales)" if os.environ.get("GIT_DIRTY") == "false" else "(¡cambios sin commit!)")
print("MLflow  :", os.environ.get("MLFLOW_TRACKING_URI"))

EXPERIMENT, DATASET, MODEL, REPORT = "E2", "v3", "models:/pettrace-embedder/2", "reports/e2_v3"
BASELINE = "reports/e1_v3/summary.json"

## 1. Datos

In [ ]:
ds = data.load(DATASET)
bucket = lambda x: "oscura" if x < 0.25 else "tenue" if x < 0.4 else "normal"
rows = ["| Especie | Animales | Fotos | Fotos por animal | Luz normal / tenue / oscura | Particiones |", "|---|---|---|---|---|---|"]
for sp in ("cat", "dog"):
    ph = [p for p in ds.photos if p.species == sp]
    per = Counter(p.animal for p in ph)
    light = Counter(bucket(p.light) for p in ph)
    splits = Counter(ds.animals[a]["split"] for a in per)
    rows.append(f"| {sp} | {len(per)} | {len(ph)} | {min(per.values())}–{max(per.values())} | "
                f"{light['normal']} / {light['tenue']} / {light['oscura']} | {dict(splits)} |")
display(Markdown(f"**{ds.name}** · commit `{ds.commit}` · huella `{ds.fingerprint}`\n\n" + "\n".join(rows)))
groups = Counter(r["lookalike_group"] for r in ds.animals.values() if r["lookalike_group"])
display(Markdown("Grupos de parecido (negativos difíciles): " + ", ".join(
    f"`{g}`: " + ", ".join(a for a, r in sorted(ds.animals.items()) if r["lookalike_group"] == g) for g in sorted(groups))))

## 2. Modelo

Misma verificación que en E1, y cómo se construyó: el `pipeline.yaml` agrega el paso `detect_crop` antes del preprocesamiento de CLIP, y la ejecución de registro guarda el detector exportado (sha256) y sus parámetros.

In [ ]:
model = load_model(MODEL)
display(Markdown("\n".join([
    "| | |", "|---|---|",
    f"| URI | `{model.uri}` (versión {model.registry_version} del registro) |",
    f"| Pipeline | `{model.model_version}` |",
    *[f"| sha256 `{f}` | `{h}` |" for f, h in model.sha256.items()],
    f"| Golden set | huella `{model.golden_fingerprint}`, {'✓ reproduce los embeddings guardados' if model.golden_ok else '✗ NO coincide'} |",
])))
assert model.golden_ok, "the downloaded model does not reproduce its golden fingerprint"

In [ ]:
# How the version was built: its own pipeline.yaml (the exact preprocessing) and its registration run.
spec = yaml.safe_load(open(os.path.join(model.dir, "pipeline.yaml")))
print(yaml.safe_dump(spec, sort_keys=False, allow_unicode=True))
client = mlflow.MlflowClient()
reg = client.get_run(client.get_model_version("pettrace-embedder", model.registry_version).run_id)
display(Markdown(
    f"Registrado por la ejecución `{reg.info.run_id[:8]}` (`{reg.info.run_name}`), script "
    f"`{os.path.basename(reg.data.tags.get('mlflow.source.name', ''))}` de este repositorio:\n\n"
    + "\n".join(["| Parámetro | Valor |", "|---|---|"] + [f"| {k} | `{v}` |" for k, v in sorted(reg.data.params.items())])
    + "\n\n" + "\n".join(["| Métrica del registro | Valor |", "|---|---|"]
                            + [f"| {k} | {v:.4g} |" for k, v in sorted(reg.data.metrics.items())])))
steps = {s["op"]: s for s in spec["steps"]}

## 3. Entrenamiento

No aplica (*zero-shot*). En su lugar se mide qué hace el detector sobre el dataset: en cuántas fotos encuentra al animal y cuánto de la imagen conserva.

In [ ]:
yolo = model.pipeline.detector
found, area, wrong_species = 0, [], 0
for p in ds.photos:
    img = ImageOps.exif_transpose(PILImage.open(p.path)).convert("RGB")
    box = yolo.box(img)
    if box is None:
        continue
    found += 1
    _, cls, (x1, y1, x2, y2) = box
    area.append(max(0, x2 - x1) * max(0, y2 - y1) / (img.width * img.height))
    wrong_species += {15: "cat", 16: "dog"}[cls] != p.species
display(Markdown(f"Animal detectado en **{found}/{len(ds.photos)}** fotos (el resto usa la imagen completa). "
                 f"El recorte conserva en mediana el **{sorted(area)[len(area)//2]:.0%}** de la imagen. "
                 f"Especie de la caja distinta a la etiqueta: {wrong_species} (no afecta: la búsqueda se filtra por la especie del reporte)."))

## 4. Evaluación, paso a paso

Todo el cálculo se hace aquí, a la vista: preprocesamiento, embeddings con ONNX Runtime, similitudes, ranking y métricas. Al final se comprueba que coincide exactamente con la librería `reid` que usan los scripts y las figuras.

### 4.1 Detección, recorte y preprocesamiento

Primero el detector (paso `detect_crop` del `pipeline.yaml`), escrito explícitamente; luego el mismo preprocesamiento de CLIP que en E1. Se comprueba en 10 fotos que el recorte manual es idéntico al del pipeline registrado.

In [ ]:
pre = steps["clip_preprocess"]
RESAMPLE = {"bicubic": PILImage.Resampling.BICUBIC, "bilinear": PILImage.Resampling.BILINEAR}

def clip_preprocess(img):
    """pipeline.yaml → clip_preprocess, written out: shortest edge → 224, centre crop 224×224,
    scale to 0–1, normalise with CLIP's mean/std, HWC → CHW."""
    short = pre["resize_shortest_edge"]
    w, h = img.size
    size = (short, int(short * h / w)) if w <= h else (int(short * w / h), short)
    img = img.resize(size, RESAMPLE[pre["resample"]])
    ch, cw = pre["center_crop"]
    left, top = (img.width - cw) // 2, (img.height - ch) // 2
    img = img.crop((left, top, left + cw, top + ch))
    x = np.asarray(img, dtype=np.float32) * pre["rescale_factor"]
    x = (x - np.array(pre["mean"], dtype=np.float32)) / np.array(pre["std"], dtype=np.float32)
    return x.transpose(2, 0, 1)

def load(photo):
    return ImageOps.exif_transpose(PILImage.open(photo.path)).convert("RGB")

In [ ]:
d = steps["detect_crop"]
detector = ort.InferenceSession(os.path.join(model.dir, d["file"]), providers=["CPUExecutionProvider"])
SIZE, CONF, MARGIN = int(d["input_size"]), float(d["conf"]), float(d["margin"])
CLASSES = [int(c) for c in d["classes"]]  # COCO 15 = gato, 16 = perro

def detect_crop(img):
    """pipeline.yaml → detect_crop, written out: letterbox to 640×640 (gray 114 padding), YOLOv8n,
    best box of a cat/dog class above the confidence, back to image pixels, 10 % margin, crop.
    No box above the threshold → the full image (fallback)."""
    w, h = img.size
    r = min(SIZE / w, SIZE / h)
    nw, nh = round(w * r), round(h * r)
    canvas = PILImage.new("RGB", (SIZE, SIZE), (114, 114, 114))
    left, top = round((SIZE - nw) / 2 - 0.1), round((SIZE - nh) / 2 - 0.1)
    canvas.paste(img.resize((nw, nh), PILImage.Resampling.BILINEAR), (left, top))
    x = np.asarray(canvas, dtype=np.float32).transpose(2, 0, 1)[None] / 255.0
    out = detector.run(None, {detector.get_inputs()[0].name: x})[0][0]  # [4 + 80 clases, anclas]
    scores = out[4:][CLASSES]
    c, a = np.unravel_index(int(np.argmax(scores)), scores.shape)
    if scores[c, a] < CONF:
        return img
    cx, cy, bw, bh = (float(v) for v in out[:4, a])
    x1, y1 = (cx - bw / 2 - left) / r, (cy - bh / 2 - top) / r
    x2, y2 = (cx + bw / 2 - left) / r, (cy + bh / 2 - top) / r
    mw, mh = MARGIN * (x2 - x1), MARGIN * (y2 - y1)
    box = (max(0, x1 - mw), max(0, y1 - mh), min(w, x2 + mw), min(h, y2 + mh))
    if box[2] - box[0] < 2 or box[3] - box[1] < 2:
        return img
    return img.crop(box)


for p in ds.photos[:10]:
    img = load(p)
    assert np.allclose(clip_preprocess(detect_crop(img)), model.pipeline.preprocess(img)), p.path
print("✓ detección + recorte + preprocesamiento manual = pipeline registrado (10 fotos)")

### 4.2 Embeddings

Cada foto preprocesada pasa por el encoder ONNX de la versión registrada (el mismo archivo, verificado por sha256, que sirve la app) y sale como un vector de 512 números.

In [ ]:
onnx_step = steps["onnx"]
encoder = ort.InferenceSession(os.path.join(model.dir, onnx_step["file"]), providers=["CPUExecutionProvider"])

def preprocess(img):
    return clip_preprocess(detect_crop(img))

p0 = ds.photos[0]
img0 = load(p0)
x0 = preprocess(img0)
print(f"ejemplo {os.path.basename(p0.path)} ({p0.animal}): {img0.size[0]}×{img0.size[1]} px → tensor {x0.shape}, "
      f"media {x0.mean():+.3f}, desviación {x0.std():.3f}")

chunks = []
for i in range(0, len(ds.photos), 32):
    X = np.stack([preprocess(load(p)) for p in ds.photos[i:i + 32]]).astype(np.float32)
    chunks.append(encoder.run([onnx_step["output"]], {onnx_step["input"]: X})[0])
E = np.concatenate(chunks)
norms = np.linalg.norm(E, axis=1)
print(f"embeddings: {E.shape[0]} fotos × {E.shape[1]} dimensiones; norma L2 entre {norms.min():.6f} y {norms.max():.6f} "
      "(el encoder ya normaliza, así que coseno = producto punto)")

### 4.3 Similitud

Como los vectores tienen norma 1, la similitud coseno entre dos fotos es su producto punto. Se calcula para todos los pares.

In [ ]:
animals = np.array([p.animal for p in ds.photos])
species = np.array([p.species for p in ds.photos])
S = E @ E.T  # similitud coseno entre todas las fotos

q = 0
same_sp = np.where((species == species[q]) & (np.arange(len(E)) != q))[0]
top = same_sp[np.argsort(-S[q, same_sp])][:5]
display(Markdown(f"Consulta de ejemplo: foto {q} de `{animals[q]}`. Las 5 fotos más parecidas de su especie:\n\n"
                 + "\n".join(["| Puesto | Animal | Similitud |", "|---|---|---|"]
                              + [f"| {i} | `{animals[g]}`{' ✓' if animals[g] == animals[q] else ''} | {S[q, g]:.3f} |"
                                 for i, g in enumerate(top, 1)])))

### 4.4 Ranking (*leave-one-out*)

Cada foto es una consulta, como un avistamiento. Se compara con todas las demás fotos **de su especie** (la propia sale de la galería); cada animal puntúa con su foto más parecida y se ordenan los animales. El *rank* es el puesto del animal correcto. Para el mAP se ordenan las fotos y se promedia la precisión en cada foto del mismo animal.

In [ ]:
# Leave-one-out within the species: the query photo leaves the gallery; each animal scores with its
# best photo (as the app does); the rank is the position of the true animal. AP ranks photos.
rows = []
for q in range(len(E)):
    gallery = np.where((species == species[q]) & (np.arange(len(E)) != q))[0]
    if len(set(animals[gallery])) < 2 or not (animals[gallery] == animals[q]).any():
        continue  # species with one animal, or an animal with a single photo: not evaluable
    best = {}
    for g in gallery:
        best[animals[g]] = max(best.get(animals[g], -np.inf), S[q, g])
    ranked = sorted(best, key=best.get, reverse=True)
    order = gallery[np.argsort(-S[q, gallery], kind="stable")]
    relevant = animals[order] == animals[q]
    precision_at_hits = np.cumsum(relevant)[relevant] / (np.flatnonzero(relevant) + 1)
    rows.append({"animal": animals[q], "species": species[q], "rank": ranked.index(animals[q]) + 1,
                 "ap": float(precision_at_hits.mean()), "ranked": ranked, "best": best})

r0 = rows[0]
display(Markdown(f"Ranking de animales para la consulta 0 (`{r0['animal']}`): "
                 + " > ".join(f"`{a}` {r0['best'][a]:.3f}" for a in r0["ranked"])
                 + f" → el animal correcto queda en el puesto **{r0['rank']}**; AP de la consulta {r0['ap']:.3f}."))

### 4.5 Métricas

**Rank-k** = proporción de consultas cuyo animal correcto está entre los k primeros. **mAP** = media de la precisión promedio por consulta. **EER** = tasa de error donde las falsas aceptaciones igualan a los falsos rechazos al variar un umbral fijo de similitud.

In [ ]:
ranks = np.array([r["rank"] for r in rows])
aps = np.array([r["ap"] for r in rows])
manual = {"global": {"queries": len(rows), "rank1": float(np.mean(ranks <= 1)), "rank5": float(np.mean(ranks <= 5)),
                     "mAP": float(aps.mean())}}
for sp in ("cat", "dog"):
    m = np.array([r["species"] == sp for r in rows])
    manual[sp] = {"queries": int(m.sum()), "rank1": float(np.mean(ranks[m] <= 1)), "rank5": float(np.mean(ranks[m] <= 5)),
                  "mAP": float(aps[m].mean())}

# EER: every pair of photos of the same species is "same animal" or "different animal"; a threshold
# t accepts a pair when its similarity ≥ t. FAR = different pairs accepted, FRR = same pairs
# rejected; the EER is where both are equal (the threshold is fitted on these same pairs: optimistic).
i, j = np.triu_indices(len(E), k=1)
keep = species[i] == species[j]
i, j = i[keep], j[keep]
same_pairs = S[i, j][animals[i] == animals[j]]
diff_pairs = S[i, j][animals[i] != animals[j]]
thresholds = np.unique(np.concatenate([same_pairs, diff_pairs]))
far = np.array([np.mean(diff_pairs >= t) for t in thresholds])
frr = np.array([np.mean(same_pairs < t) for t in thresholds])
k = int(np.argmin(np.abs(far - frr)))
manual_eer, manual_thr = (far[k] + frr[k]) / 2, thresholds[k]

display(Markdown("\n".join(["| | Consultas | Rank-1 | Rank-5 | mAP |", "|---|---|---|---|---|"]
    + [f"| {g} | {v['queries']} | {v['rank1']:.1%} | {v['rank5']:.1%} | {v['mAP']:.3f} |" for g, v in manual.items()])
    + f"\n\nPares: {len(same_pairs)} del mismo animal y {len(diff_pairs)} de animales distintos. "
    f"**EER {manual_eer:.1%}** con umbral {manual_thr:.3f}."))

### 4.6 Verificación

In [ ]:
# The same evaluation through the library (reid.evaluation / reid.metrics, used by the CLI and the
# figures below) must give exactly what was computed by hand above.
ev = evaluate(ds, model.pipeline, zero_shot=True)
lib = ev.result["test"]
assert np.allclose(E, ev.embeddings, atol=1e-6), "embeddings differ"
assert [r["rank"] for r in rows] == [r["rank"] for r in lib["per_query"]], "ranks differ"
for g in ("global", "cat", "dog"):
    for key in ("rank1", "rank5", "mAP"):
        assert abs(manual[g][key] - lib[g][key]) < 1e-9, (g, key)
assert abs(manual_eer - ev.result["val"]["eer"]) < 1e-9 and abs(manual_thr - ev.result["val"]["threshold"]) < 1e-9
print(f"✓ el cálculo manual coincide con la librería: embeddings (máx |Δ| {np.abs(E - ev.embeddings).max():.1e}), "
      f"{len(rows)} rankings, Rank-1/5, mAP y EER idénticos")

## 5. Resultados y comparación con E1

In [ ]:
s2 = save(ev, model, REPORT)
s1 = json.load(open(BASELINE))
display(Markdown(metrics_table({"E1": s1, "E2": s2})))

In [ ]:
summ = json.load(open(os.path.join(REPORT, "summary.json")))
for name in ("confusion", "rank1_por_animal", "distribucion_scores", "cmc", "matriz_similitud"):
    display(Image(os.path.join(REPORT, f"{name}.png"), width=760))
    display(Markdown(explain(name, summ)))

Confusión de E1 (línea base) para comparar con la de E2:

In [ ]:
display(Image("reports/e1_v3/confusion.png", width=760))

## 6. Registro en MLflow

In [ ]:
run_id = log_run(ev, model, experiment=EXPERIMENT, artifacts=REPORT)
json.dump({"run_id": run_id, "experiment": EXPERIMENT, "dataset": ds.name, "dataset_commit": ds.commit,
           "model": MODEL, "git_sha": os.environ.get("GIT_SHA", "unknown")}, open(os.path.join(REPORT, "run.json"), "w"), indent=2)
display(Markdown(f"Ejecución MLflow: [`{run_id[:8]}`]({os.environ['MLFLOW_TRACKING_URI']}/#/experiments/"
                 f"{mlflow.get_experiment_by_name('pettrace-reid-eval').experiment_id}/runs/{run_id})"))

## 7. Conclusiones

- **El recorte separa mejor las similitudes**: baja la EER y sube el mAP respecto de E1, y la similitud mínima entre fotos del mismo animal aumenta. En perros también mejora el Rank-1.
- **En gatos el Rank-1 baja**, y la caída se concentra en los grupos de parecido: `cat-01` se confunde con `cat-02` (mismo patrón atigrado) en 6 de sus 10 consultas. Sin el fondo, el modelo pierde pistas de entorno que ayudaban a E1 y el pelaje restante es casi idéntico.
- Por lo tanto, **parte del acierto de E1 venía del contexto de la foto**, que no existe cuando el avistamiento ocurre en la calle: E2 mide la identidad de forma más honesta y muestra que CLIP sin especializar no distingue gatos de pelaje parecido.
- Con 128 consultas la diferencia de Rank-1 entre E1 y E2 está dentro del ruido (intervalos de confianza pendientes). E2 queda como base para E3–E5: especializar el encoder con el dataset propio y re-ordenar con la descripción (p. ej. «le falta un ojo» para `cat-02`).